In [ ]:
%load_ext autoreload

In [ ]:
import itertools
from pathlib import Path
import re
from typing import Literal

from matplotlib.colors import CenteredNorm
from matplotlib import transforms
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pickle
from scipy import stats
from scipy.stats import ttest_ind, pearsonr, spearmanr
import seaborn as sns
import textgrid
import torch
from tqdm.auto import tqdm
tqdm.pandas()

In [ ]:
%autoreload 2

from src.data import get_electrode_df, add_metadata_features
from src.models.causal4 import run_causal4_analysis
from src.models import causal4
import src.viz as viz

In [ ]:
epochs_paths = list(Path("outputs/epochs_preprocessed").glob("*.fif"))

tg_dir = "textgrids"

timit_epoch_sources = {
    "All": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-all.h5",
    "Word onset": "/userdata/jgauthier/projects/ideal-word-representations/phoneme_epochs-onsets.h5",
}

electrodes_paths = list(Path("outputs/causal4/find_speech_responsive").glob("*_results.csv"))

# all_A_result_paths = list(Path("outputs/causal4/find_As").glob("*_results.csv"))
# all_A_decoder_paths = list(Path("outputs/causal4/find_As").glob("*_decoders.pt"))
A_result_path = Path("outputs/causal4/unify_As/results.csv")
A_decoders_path = Path("outputs/causal4/unify_As/unified_decoders.pt")
all_B_result_paths = list(Path("outputs/causal4/find_Bs").glob("*_results.csv"))

outdir = "."

## Load results

In [ ]:
epochs = {
    re.search(r"(\w+)_epo.fif", str(path)).group(1): mne.read_epochs(path, preload=True, verbose=False)
    for path in epochs_paths
}

In [ ]:
for e in epochs.values():
    e.metadata = add_metadata_features(e.metadata)

In [ ]:
electrode_df = pd.concat([pd.read_csv(path) for path in electrodes_paths]).set_index(["subject", "electrode_idx"])

In [ ]:
A_results = pd.read_csv(A_result_path)
B_results = pd.concat(
    [pd.read_csv(path) for path in all_B_result_paths],
    ignore_index=True,
)

In [ ]:
A_decoders = torch.load(A_decoders_path)

In [ ]:
B_decoders = {
    re.search(r"([\w]+)_decoders.pt", str(path)).group(1): torch.load(path)
    for path in all_B_decoder_paths
}

In [ ]:
assert set(epochs.keys()) == set(A_decoders["train_scores"].subject.unique())
assert set(epochs.keys()) == set(A_results.subject)
assert set(epochs.keys()) == set(B_results.subject)
subjects = sorted(A_results.subject.unique())

## Prepare to plot

In [ ]:
timit_bounds = viz.precompute_timit_bounds(timit_epoch_sources, subjects=subjects)

In [ ]:
plot_B_site(B_results.sort_values("p_val_min").iloc[0])
None

In [ ]:
subject, plot_meta_df, population_B_window = dev[0]

g_raster = causal4.plot_causal4_raster(
    epochs[subject], plot_meta_df, subject, population_B_window,
    sort_by="p_gt_phoneme", plot_extremes=True, parameter_cache=parameter_cache, cbar=False)

In [ ]:
subject, plot_meta_df, population_B_window = dev[0]

g_raster = causal4.plot_causal4_raster(
    epochs[subject], plot_meta_df, subject, population_B_window,
    sort_by="resampled", plot_extremes=True, parameter_cache=parameter_cache, cbar=False)

In [ ]:
# cache z-score parameters for each subject and electrode
parameter_cache = {}
dev = []

def plot_B_site(row):
    subject = row.subject
    phoneme_pair = row.phoneme_pair
    population_A = row.population_name
    population_B = [row.electrode_idx]
    left = row.left_is_best
    population_B_window = (int(row.window_start_samp), int(row.window_end_samp))

    A_row = A_results[(A_results.subject == subject) & (A_results.phoneme_pair == phoneme_pair) & (A_results.population_name == population_A)]
    assert len(A_row) == 1, f"Expected one row for {subject} {phoneme_pair} {population_A}, got {len(A_row)}"

    plot_key = (subject, phoneme_pair, population_A)
    plot_num_quantiles = 4
    epochs_i = epochs[subject]

    # convert to samples
    population_A_window = (
        epochs_i.time_as_index(A_row.smin)[0],
        epochs_i.time_as_index(A_row.smax)[0],
    )

    A_outcomes = A_decoders["held_out_outcomes"][subject, population_A, phoneme_pair]

    # sanity check: test trials are the same across repeats
    test_trial_indices = A_outcomes.groupby("fold").apply(lambda xs: xs.epoch_idx.unique()).values
    for i in range(1, len(test_trial_indices)):
        np.testing.assert_array_equal(
            test_trial_indices[i],
            test_trial_indices[0],
        )

    # merge estimated probabilities from repeats
    plot_meta_df = pd.merge(
        A_outcomes.groupby("epoch_idx").decoder_proba.mean().reset_index(),
        epochs[subject].metadata,
        how="left", left_on="epoch_idx", right_index=True
    )

    # p(gt phoneme) is decoder probability if we are looking at the right phoneme,
    # or 1 - decoder probability if we are looking at the left phoneme
    plot_meta_df["p_gt_phoneme"] = plot_meta_df.groupby("label_lexical").decoder_proba.transform(
        lambda xs: 1 - xs if xs.name == phoneme_pair[0] else xs)
    plot_meta_df["p_gt_phoneme_binned"] = pd.qcut(
        plot_meta_df.p_gt_phoneme, plot_num_quantiles,
        # labels=[f"Q{i+1}" for i in range(plot_num_quantiles)]
    )
    plot_meta_df["p_gt_phoneme_bin_center"] = plot_meta_df.p_gt_phoneme_binned.apply(
        lambda x: x.mid
    ).astype(float).round(3)

    # TODO concat outcomes from extremes

    def get_textgrid_path(row):
        return Path(tg_dir) / (Path(row.wav_file).with_suffix(".TextGrid").name)
    plot_meta_df["textgrid_path"] = plot_meta_df.apply(get_textgrid_path, axis=1)

    # cross by electrodes
    plot_meta_df = pd.merge(plot_meta_df,
                 electrode_df.loc[subject].loc[population_B].reset_index(),
                 how="cross")

    ####

    g_scatter = causal4.plot_causal4_scatter(
        epochs[subject], plot_meta_df, subject, population_B_window)

    ####

    # # displot showing spearmanr results
    # spearmanr_results = np.array(row.counterfactual_spearmanr_list)
    # g_displot = sns.displot(
    #     spearmanr_results,
    #     kind="kde", fill=True, color="blue",
    #     height=2.5, aspect=3,
    # )
    # spearmanr_obs = row.corr_left if row.p_val_left < row.p_val_right else row.corr_right
    # g_displot.ax.axvline(spearmanr_obs, color="red", linestyle="--", label="Observed", linewidth=2)
    # g_displot.ax.set_xlabel("Spearman correlation")
    # g_displot.ax.set_title(f"Permutation baseline results\n(z={row.counterfactual_test_z:.2f}, p={row.counterfactual_test_p:.2g})")
    g_displot = None

    ####

    g = causal4.plot_causal4_evoked(
        epochs[subject], plot_meta_df, subject, population_A_window, population_B_window,
        hue="p_gt_phoneme_bin_center"
    )

    ####

    # # plot re-aligned to behavior
    # def plot_facet_evoked_align_behavior(data, color, **kwargs):
    #     electrode_idx = data.electrode_idx.iloc[0]
    #     epoch_idxs = data.epoch_idx

    #     word_end = data.word_end.iloc[0]
    #     tg_path = data.textgrid_path.iloc[0]
    #     tg = textgrid.TextGrid.fromFile(str(tg_path))

    #     ax = plt.gca()
    #     ax.set_xlabel("Time relative to behavior onset (sec)")
    #     ax.set_ylabel("HGA")
    #     ax.set_title(f"{subject} {electrode_idx + 1}, {word_end}")

    #     # plot epoched response at this electrode
    #     plot_epochs = epochs[subject][epoch_idxs]
    #     plot_epochs = causal4.realign_epochs_by_behavior(plot_epochs)

    #     plot_epoch_data = plot_epochs.copy().pick(electrode_idx).get_data().squeeze(1)
    #     assert plot_epoch_data.ndim == 2  # n_trials * n_times

    #     plot_times = plot_epochs.times
    #     plot_epoch_data_mean = np.nanmean(plot_epoch_data, 0)
    #     plot_epoch_data_sem = np.nanstd(plot_epoch_data, 0) / np.sqrt((~np.isnan(plot_epoch_data)).sum(0))
    #     ax.plot(plot_times, plot_epoch_data_mean, color=color, alpha=0.5, **kwargs)
    #     ax.fill_between(plot_times, plot_epoch_data_mean - plot_epoch_data_sem,
    #                     plot_epoch_data_mean + plot_epoch_data_sem, color=color, alpha=0.2)
        
    #     ax.set_xlim(plot_epochs.times[0], plot_epochs.times[-1])

    #     return ax
    
    # g_evoked_by_behavior = sns.FacetGrid(
    #     plot_meta_df,
    #     row="electrode_idx",
    #     hue="p_gt_phoneme_bin_center", palette="plasma",
    #     col="lexical_evidence",
    #     aspect=3, height=3, sharey="row"
    # ).map_dataframe(plot_facet_evoked_align_behavior).add_legend()
    # g_evoked_by_behavior.fig.suptitle("Evoked responses by P(gt phoneme), aligned to behavior onset")
    g_evoked_by_behavior = None

    ####

    g_evoked_resampled = causal4.plot_causal4_evoked(
        epochs[subject], plot_meta_df, subject, population_A_window, population_B_window,
        hue="resampled"
    )

    ####

    g_raster = causal4.plot_causal4_raster(
        epochs[subject], plot_meta_df, subject, population_B_window,
        plot_extremes=True,
        sort_by="p_gt_phoneme", parameter_cache=parameter_cache)
    
    dev.append((subject, plot_meta_df, population_B_window))
    g_raster_resampled = causal4.plot_causal4_raster(
        epochs[subject], plot_meta_df, subject, population_B_window,
        sort_by="resampled", parameter_cache=parameter_cache, cbar=False)
    
    g_raster_behavior = causal4.plot_causal4_raster(
        epochs[subject], plot_meta_df, subject, population_B_window,
        sort_by="behavior_linear", parameter_cache=parameter_cache, cbar=False)

    ####

    timit_fig = viz.timit_subplots(
        subject, population_B[0],
        plot_phonemes=[ph.upper() for ph in phoneme_pair],
        cell_aspect=1.5,
        epoch_sources=timit_epoch_sources,
        timit_bounds_dict=timit_bounds)
    timit_fig.suptitle(f"TIMIT responses for {subject} {population_B[0] + 1}")

    return (g_scatter, g_displot,
            g, g_evoked_by_behavior, g_evoked_resampled,
            g_raster, g_raster_resampled, g_raster_behavior,
            timit_fig)